# FlowVAE con Pyro -- entrenamiento en Colab (GPU)

Este notebook entrena el `FlowVAE` (embeddings por columna categorica + Pyro SVI/Trace_ELBO)
sobre los datos de flujos de red ya preprocesados localmente (`train.npz`, `test.npz`, `metadata.json`).

**Antes de correr:** anda a `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion` y elegi **GPU** (T4).

Pasos:
1. Instalar Pyro.
2. Subir `train.npz`, `test.npz` y `metadata.json` (generados por `preprocess.py` en tu maquina).
3. Correr el entrenamiento.
4. Descargar los checkpoints y la curva de perdida.


In [ ]:
!pip install pyro-ppl -q


## 1. Subir los datos preprocesados

Subi `train.npz`, `test.npz` y `metadata.json` (carpeta `processed/` de tu maquina).

In [ ]:
from google.colab import files
import os

os.makedirs("processed", exist_ok=True)
print("Subi train.npz, test.npz y metadata.json")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, os.path.join("processed", name))
print("Listo:", os.listdir("processed"))


### Alternativa: Google Drive

Si preferis no re-subir los archivos cada vez que se reinicia la sesion, monta Drive y
apunta `PROCESSED_DIR` a la carpeta correspondiente en vez de usar el upload de arriba:

```python
from google.colab import drive
drive.mount('/content/drive')
PROCESSED_DIR = '/content/drive/MyDrive/Modelo_VAE/processed'
```


In [ ]:
import json
import csv
from pathlib import Path

import numpy as np
import torch
import pyro
import pyro.distributions as dist
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

PROCESSED_DIR = Path("processed")
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Dataset: carga vectorizada de train/test (sin Dataset/DataLoader fila-por-fila)

In [ ]:
class FlowTensors:
    """Carga train/test enteros en tensores y arma batches con indexado
    vectorizado (torch.index_select), evitando el overhead de Dataset/
    DataLoader fila-por-fila que con millones de filas tardaria horas."""

    def __init__(self, npz_path, categorical_cols, device):
        data = np.load(npz_path)
        self.numeric = torch.from_numpy(data["numeric"]).float().to(device)
        self.categorical = {
            c: torch.from_numpy(data[f"cat__{c}"]).long().to(device) for c in categorical_cols
        }
        self.n = self.numeric.shape[0]
        self.categorical_cols = categorical_cols

    def batches(self, batch_size, shuffle, device):
        idx = torch.randperm(self.n, device=device) if shuffle else torch.arange(self.n, device=device)
        for start in range(0, self.n, batch_size):
            b = idx[start:start + batch_size]
            x_cat = {c: self.categorical[c].index_select(0, b) for c in self.categorical_cols}
            x_num = self.numeric.index_select(0, b)
            yield x_cat, x_num


## 3. Modelo: FlowVAE (Pyro) -- identico al de la maquina local (`model.py`)

In [ ]:
class CategoricalEmbedder(nn.Module):
    def __init__(self, categorical_cols, vocab_sizes, embedding_dims, sparse_cols=()):
        super().__init__()
        self.categorical_cols = categorical_cols
        self.sparse_cols = set(sparse_cols)
        self.embeddings = nn.ModuleDict({
            c: nn.Embedding(vocab_sizes[c], embedding_dims[c], padding_idx=0, sparse=(c in self.sparse_cols))
            for c in categorical_cols
        })
        self.output_dim = sum(embedding_dims[c] for c in categorical_cols)

    def forward(self, x_cat):
        return torch.cat([self.embeddings[c](x_cat[c]) for c in self.categorical_cols], dim=-1)


class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.loc = nn.Linear(hidden_dim, latent_dim)
        self.log_scale = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.net(x)
        z_loc = self.loc(h)
        z_scale = torch.clamp(self.log_scale(h), -10, 10).exp()
        return z_loc, z_scale


class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, decoder_categorical_cols, vocab_sizes, n_numeric):
        super().__init__()
        self.decoder_categorical_cols = decoder_categorical_cols
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.numeric_head = nn.Linear(hidden_dim, n_numeric)
        self.numeric_log_scale = nn.Parameter(torch.zeros(n_numeric))
        self.categorical_heads = nn.ModuleDict({
            c: nn.Linear(hidden_dim, vocab_sizes[c]) for c in decoder_categorical_cols
        })

    def forward(self, z):
        h = self.net(z)
        num_recon = torch.sigmoid(self.numeric_head(h))
        cat_logits = {c: self.categorical_heads[c](h) for c in self.decoder_categorical_cols}
        return num_recon, cat_logits


class FlowVAE(nn.Module):
    def __init__(self, categorical_cols, numeric_cols, vocab_sizes, embedding_dims,
                 hidden_dim=128, latent_dim=16, max_decoder_vocab=10_000):
        super().__init__()
        self.categorical_cols = categorical_cols
        self.numeric_cols = numeric_cols
        self.n_numeric = len(numeric_cols)
        self.latent_dim = latent_dim

        # columnas de cardinalidad manejable: se reconstruyen en el decoder.
        # Las de cardinalidad grande (dst_ip, src_port) solo se embeben para
        # el encoder -- ninguna variable se descarta, solo no tienen cabeza
        # de clasificacion en el decoder (softmax completo sobre millones de
        # clases no es viable).
        self.decoder_categorical_cols = [c for c in categorical_cols if vocab_sizes[c] <= max_decoder_vocab]
        self.encoder_only_cols = [c for c in categorical_cols if vocab_sizes[c] > max_decoder_vocab]

        self.embedder = CategoricalEmbedder(categorical_cols, vocab_sizes, embedding_dims,
                                             sparse_cols=self.encoder_only_cols)
        input_dim = self.embedder.output_dim + self.n_numeric
        self.input_dim = input_dim

        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, self.decoder_categorical_cols, vocab_sizes, self.n_numeric)

    def parameter_groups(self):
        sparse_params = [self.embedder.embeddings[c].weight for c in self.embedder.sparse_cols]
        sparse_ids = {id(p) for p in sparse_params}
        dense_params = [p for p in self.parameters() if id(p) not in sparse_ids]
        return dense_params, sparse_params

    def model(self, x_cat, x_num):
        pyro.module("decoder", self.decoder)
        batch_size = x_num.shape[0]
        with pyro.plate("data", batch_size):
            z_loc = x_num.new_zeros(batch_size, self.latent_dim)
            z_scale = x_num.new_ones(batch_size, self.latent_dim)
            z = pyro.sample("latent", dist.Normal(z_loc, z_scale).to_event(1))

            num_recon, cat_logits = self.decoder(z)
            numeric_scale = self.decoder.numeric_log_scale.exp()
            pyro.sample("obs_numeric", dist.Normal(num_recon, numeric_scale).to_event(1), obs=x_num)

            for c in self.decoder.decoder_categorical_cols:
                pyro.sample(f"obs_{c}", dist.Categorical(logits=cat_logits[c]), obs=x_cat[c])

    def guide(self, x_cat, x_num):
        pyro.module("embedder", self.embedder)
        pyro.module("encoder", self.encoder)
        batch_size = x_num.shape[0]
        with pyro.plate("data", batch_size):
            x_embedded = self.embedder(x_cat)
            x_in = torch.cat([x_embedded, x_num], dim=-1)
            z_loc, z_scale = self.encoder(x_in)
            pyro.sample("latent", dist.Normal(z_loc, z_scale).to_event(1))

    def encode(self, x_cat, x_num):
        x_embedded = self.embedder(x_cat)
        x_in = torch.cat([x_embedded, x_num], dim=-1)
        return self.encoder(x_in)

    @torch.no_grad()
    def reconstruction_error(self, x_cat, x_num):
        z_loc, _ = self.encode(x_cat, x_num)
        num_recon, _ = self.decoder(z_loc)
        return ((num_recon - x_num) ** 2).mean(dim=-1)


## 4. Optimizador mixto (Adam denso + SparseAdam para columnas de cardinalidad grande)

In [ ]:
def build_pyro_optimizer(vae, lr):
    """Pyro crea un optimizador por cada tensor de parametro. Los embeddings
    sparse (dst_ip, src_port) necesitan SparseAdam en vez de Adam denso --
    si no, el optimizador actualizaria la tabla ENTERA en cada paso sin
    importar cuantas filas del batch la usan."""
    _, sparse_params = vae.parameter_groups()
    sparse_ids = {id(p) for p in sparse_params}

    def optim_constructor(params, **kwargs):
        p = params[0]
        if id(p) in sparse_ids:
            return torch.optim.SparseAdam(params, lr=kwargs.get("lr", lr))
        return torch.optim.Adam(params, **kwargs)

    return pyro.optim.PyroOptim(optim_constructor, {"lr": lr})


def run_epoch(svi, tensors, batch_size, train, device):
    total_loss = 0.0
    total_rows = 0
    for x_cat, x_num in tensors.batches(batch_size, shuffle=train, device=device):
        if train:
            loss = svi.step(x_cat, x_num)
        else:
            loss = svi.evaluate_loss(x_cat, x_num)
        total_loss += loss
        total_rows += x_num.shape[0]
    return total_loss / total_rows


## 5. Hiperparametros

In [ ]:
EPOCHS = 20
BATCH_SIZE = 4096
HIDDEN_DIM = 128
LATENT_DIM = 16
LR = 1e-3
MAX_DECODER_VOCAB = 10_000


## 6. Entrenamiento

In [ ]:
with open(PROCESSED_DIR / "metadata.json") as f:
    metadata = json.load(f)

categorical_cols = metadata["categorical_cols"]
numeric_cols = metadata["numeric_cols"]
vocab_sizes = metadata["vocab_sizes"]
embedding_dims = metadata["embedding_dims"]

train_tensors = FlowTensors(PROCESSED_DIR / "train.npz", categorical_cols, device)
test_tensors = FlowTensors(PROCESSED_DIR / "test.npz", categorical_cols, device)

pyro.clear_param_store()
vae = FlowVAE(
    categorical_cols=categorical_cols,
    numeric_cols=numeric_cols,
    vocab_sizes=vocab_sizes,
    embedding_dims=embedding_dims,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    max_decoder_vocab=MAX_DECODER_VOCAB,
).to(device)

print(f"Dim. de entrada al encoder (embeddings + numericas escaladas): {vae.input_dim}")
if vae.encoder_only_cols:
    print(f"Columnas solo-encoder (siguen embebidas, no se reconstruyen en el decoder): {vae.encoder_only_cols}")

optimizer = build_pyro_optimizer(vae, LR)
svi = pyro.infer.SVI(vae.model, vae.guide, optimizer, loss=pyro.infer.Trace_ELBO())

history_path = CHECKPOINT_DIR / "loss_history.csv"
plot_path = CHECKPOINT_DIR / "loss_curve.png"

def checkpoint_payload(args_dict):
    return {
        "embedder": vae.embedder.state_dict(),
        "encoder": vae.encoder.state_dict(),
        "decoder": vae.decoder.state_dict(),
        "args": args_dict,
        "metadata": metadata,
    }

args_dict = dict(epochs=EPOCHS, batch_size=BATCH_SIZE, hidden_dim=HIDDEN_DIM,
                  latent_dim=LATENT_DIM, lr=LR, max_decoder_vocab=MAX_DECODER_VOCAB)

history = []
best_test_loss = float("inf")

with open(history_path, "w", newline="") as f:
    csv.writer(f).writerow(["epoch", "train_loss", "test_loss"])

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(svi, train_tensors, BATCH_SIZE, train=True, device=device)
    test_loss = run_epoch(svi, test_tensors, BATCH_SIZE, train=False, device=device)
    print(f"epoch {epoch:3d}  train loss (-ELBO/obs) = {train_loss:.4f}  test loss (-ELBO/obs) = {test_loss:.4f}")

    history.append((epoch, train_loss, test_loss))
    with open(history_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, train_loss, test_loss])

    torch.save(checkpoint_payload(args_dict), CHECKPOINT_DIR / "vae_last.pt")
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        torch.save(checkpoint_payload(args_dict), CHECKPOINT_DIR / "vae_best.pt")
        print(f"  -> nuevo mejor test loss ({test_loss:.4f}), guardado en vae_best.pt")

    epochs_, train_losses, test_losses = zip(*history)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs_, train_losses, label="train")
    plt.plot(epochs_, test_losses, label="test")
    plt.xlabel("epoch")
    plt.ylabel("loss (-ELBO / obs)")
    plt.title("FlowVAE - curva de perdida")
    plt.legend()
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.close()

print(f"Checkpoints en {CHECKPOINT_DIR} (vae_last.pt, vae_best.pt)")
display(Image(filename=str(plot_path)))


## 7. Descargar resultados

In [ ]:
from google.colab import files

files.download(str(CHECKPOINT_DIR / "vae_best.pt"))
files.download(str(CHECKPOINT_DIR / "vae_last.pt"))
files.download(str(CHECKPOINT_DIR / "loss_history.csv"))
files.download(str(CHECKPOINT_DIR / "loss_curve.png"))
